# 2. Document Parsing

This notebook demonstrates two core parsing approaches:

1. **AI_PARSE** - Databricks native AI-powered document parsing (simplest)
2. **Docling Single** - Single document processing with native Docling (most flexible)

Both methods will populate the same Delta tables for use by the chat application.

In [ ]:
# Setup and imports
import pandas as pd
from pathlib import Path
import json
import time
from databricks.sdk import WorkspaceClient

# Configuration
CATALOG = "main"
SCHEMA = "default"
RAW_DOCS_VOL = "raw_docs"
PROCESSED_DOCS_VOL = "processed_docs"
DOCUMENTS_TABLE = f"{CATALOG}.{SCHEMA}.processed_documents"
CHUNKS_TABLE = f"{CATALOG}.{SCHEMA}.document_chunks"

w = WorkspaceClient()

print("🚀 Document Parsing Notebook Ready!")
print(f"Input: /Volumes/{CATALOG}/{SCHEMA}/{RAW_DOCS_VOL}")
print(f"Output: /Volumes/{CATALOG}/{SCHEMA}/{PROCESSED_DOCS_VOL}")

## Method 1: AI_PARSE (Databricks Native)

AI_PARSE is Databricks' native AI-powered document parsing function. It's the simplest approach and requires no additional setup.

In [ ]:
# AI_PARSE Example
from databricks.connect import DatabricksSession
from pyspark.sql import functions as F

# Get available documents
raw_doc_dir = f"/Volumes/{CATALOG}/{SCHEMA}/{RAW_DOCS_VOL}"
available_docs = list(Path(raw_doc_dir).glob("*.pdf"))

print(f"📂 Found {len(available_docs)} PDF documents:")
for doc in available_docs:
    print(f"  📄 {doc.name}")

if available_docs:
    # Process first document with AI_PARSE
    sample_doc = str(available_docs[0])
    print(f"\n🔍 Processing with AI_PARSE: {Path(sample_doc).name}")
    
    # Create Spark session
    spark = DatabricksSession.builder.getOrCreate()
    
    # Use AI_PARSE function
    result_df = spark.sql(f"""
        SELECT 
            ai_parse('{sample_doc}') as parsed_content,
            '{sample_doc}' as doc_path,
            '{Path(sample_doc).name}' as file_name
    """)
    
    # Show results
    parsed_result = result_df.collect()[0]
    parsed_content = parsed_result['parsed_content']
    
    print("✅ AI_PARSE completed!")
    print(f"📊 Parsed content length: {len(str(parsed_content))}")
    
    # Display sample of parsed content
    print("\n📝 Sample parsed content:")
    print(str(parsed_content)[:500] + "..." if len(str(parsed_content)) > 500 else str(parsed_content))
    
else:
    print("❌ No PDF documents found. Please run 1_setup.ipynb first.")

In [ ]:
# Store AI_PARSE results in tables
if available_docs and 'parsed_content' in locals():
    
    def populate_tables_aiparse(doc_path, file_name, parsed_content):
        """Populate tables with AI_PARSE results."""
        try:
            # Insert document record
            doc_sql = f"""
            INSERT OR REPLACE INTO {DOCUMENTS_TABLE} VALUES (
                '{doc_path}',
                '{file_name}',
                'aiparse',
                0, -- pages (not available from AI_PARSE)
                0, -- pictures 
                0, -- tables
                0, -- pictures_with_descriptions
                {len(str(parsed_content))}, -- main_text_length
                false, -- vlm_enabled
                '', -- output_location
                map(), -- processing_options
                current_timestamp(),
                current_timestamp(),
                'completed'
            )
            """
            
            w.statement_execution.execute_statement(
                warehouse_id="your_warehouse_id", 
                statement=doc_sql,
                wait_timeout="30s"
            )
            
            # Create chunks from parsed content
            content_str = str(parsed_content)
            chunk_size = 1000  # Characters per chunk
            
            for i, chunk_start in enumerate(range(0, len(content_str), chunk_size)):
                chunk_text = content_str[chunk_start:chunk_start + chunk_size]
                
                chunk_sql = f"""
                INSERT INTO {CHUNKS_TABLE} VALUES (
                    '{doc_path}',
                    '{file_name}',
                    {i},
                    0, -- page_number (not available)
                    '{chunk_text.replace("'", "''")}',
                    'text',
                    {len(chunk_text.split())}, -- token_count
                    '', -- image_path
                    '', -- vlm_description
                    map(), -- metadata
                    current_timestamp()
                )
                """
                
                w.statement_execution.execute_statement(
                    warehouse_id="your_warehouse_id",
                    statement=chunk_sql,
                    wait_timeout="10s"
                )
            
            print(f"✅ Populated tables for {file_name} ({i+1} chunks)")
            
        except Exception as e:
            print(f"❌ Error populating tables: {e}")
    
    # Populate tables with AI_PARSE results
    populate_tables_aiparse(sample_doc, Path(sample_doc).name, parsed_content)

## Method 2: Docling Single Document Processing

Docling provides more detailed document analysis including layout detection, table extraction, and image processing.

In [ ]:
# Install Docling if needed
%pip install docling>=2.68.0

In [ ]:
# Docling processing
from docling.document_converter import DocumentConverter
from docling.datamodel.pipeline_options import PdfPipelineOptions

# Initialize converter with basic options
pdf_options = PdfPipelineOptions(do_ocr=True, do_table_structure=True)
converter = DocumentConverter(
    pdf_pipeline_options=pdf_options
)

print("🔧 Docling converter initialized")

if available_docs:
    # Process first document with Docling
    sample_doc = available_docs[0] 
    print(f"\n🔍 Processing with Docling: {sample_doc.name}")
    
    start_time = time.time()
    
    # Convert document
    result = converter.convert(source=sample_doc)
    document = result.document
    
    processing_time = time.time() - start_time
    
    # Extract information
    pages = len(document.pages)
    pictures = len(document.pictures) 
    tables = len(document.tables)
    main_text = document.export_to_markdown()
    
    print(f"✅ Docling processing completed in {processing_time:.2f}s")
    print(f"📊 Results:")
    print(f"  📄 Pages: {pages}")
    print(f"  🖼️  Pictures: {pictures}")
    print(f"  📋 Tables: {tables}")
    print(f"  📝 Text length: {len(main_text):,} characters")
    
    # Save outputs
    output_dir = Path(f"/Volumes/{CATALOG}/{SCHEMA}/{PROCESSED_DOCS_VOL}") / sample_doc.stem
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Save as JSON and Markdown
    document.save_as_json(output_dir / "document.json")
    document.save_as_markdown(output_dir / "document.md")
    
    print(f"💾 Saved outputs to: {output_dir}")
    
else:
    print("❌ No documents available for processing")

In [ ]:
# Store Docling results in tables
if available_docs and 'document' in locals():
    
    def populate_tables_docling(doc_path, file_name, document, output_location):
        """Populate tables with Docling results."""
        try:
            # Insert document record
            doc_sql = f"""
            INSERT OR REPLACE INTO {DOCUMENTS_TABLE} VALUES (
                '{doc_path}',
                '{file_name}',
                'docling_single',
                {len(document.pages)},
                {len(document.pictures)},
                {len(document.tables)},
                0,
                {len(document.export_to_markdown())},
                false,
                '{output_location}',
                map('do_ocr', 'true', 'do_table_structure', 'true'),
                current_timestamp(),
                current_timestamp(),
                'completed'
            )
            """
            
            w.statement_execution.execute_statement(
                warehouse_id="your_warehouse_id",
                statement=doc_sql,
                wait_timeout="30s"
            )
            
            # Create chunks from pages
            chunk_idx = 0
            for page_idx, page in enumerate(document.pages):
                page_text = page.export_to_markdown()
                
                # Split page into manageable chunks
                chunk_size = 1000
                for chunk_start in range(0, len(page_text), chunk_size):
                    chunk_text = page_text[chunk_start:chunk_start + chunk_size]
                    
                    if chunk_text.strip():  # Only insert non-empty chunks
                        chunk_sql = f"""
                        INSERT INTO {CHUNKS_TABLE} VALUES (
                            '{doc_path}',
                            '{file_name}',
                            {chunk_idx},
                            {page_idx + 1},
                            '{chunk_text.replace("'", "''")}',
                            'text',
                            {len(chunk_text.split())},
                            '',
                            '',
                            map('source', 'docling'),
                            current_timestamp()
                        )
                        """
                        
                        w.statement_execution.execute_statement(
                            warehouse_id="your_warehouse_id",
                            statement=chunk_sql,
                            wait_timeout="10s"
                        )
                        
                        chunk_idx += 1
            
            print(f"✅ Populated tables for {file_name} ({chunk_idx} chunks)")
            
        except Exception as e:
            print(f"❌ Error populating tables: {e}")
    
    # Populate tables with Docling results
    populate_tables_docling(
        str(sample_doc), 
        sample_doc.name, 
        document, 
        str(output_dir)
    )

## Processing Summary

View the results of your document processing:

In [ ]:
# Query processed documents
try:
    # Get document summary
    docs_query = f"SELECT * FROM {DOCUMENTS_TABLE} ORDER BY created_at DESC LIMIT 10"
    response = w.statement_execution.execute_statement(
        warehouse_id="your_warehouse_id",
        statement=docs_query,
        wait_timeout="30s"
    )
    
    if response.result and response.result.data_array:
        print("📊 Recently Processed Documents:")
        print("-" * 80)
        for row in response.result.data_array:
            file_name = row[1]
            method = row[2] 
            pages = row[3]
            pictures = row[4]
            tables = row[5]
            text_len = row[7]
            status = row[13]
            
            print(f"📄 {file_name}")
            print(f"   Method: {method}")
            print(f"   Content: {pages} pages, {pictures} pictures, {tables} tables")
            print(f"   Text: {text_len:,} characters")
            print(f"   Status: {status}")
            print()
    
    # Get chunk summary
    chunks_query = f"SELECT COUNT(*) as total_chunks, SUM(token_count) as total_tokens FROM {CHUNKS_TABLE}"
    response = w.statement_execution.execute_statement(
        warehouse_id="your_warehouse_id",
        statement=chunks_query,
        wait_timeout="30s"
    )
    
    if response.result and response.result.data_array:
        row = response.result.data_array[0]
        total_chunks = row[0]
        total_tokens = row[1]
        
        print(f"📈 Processing Statistics:")
        print(f"   Total chunks: {total_chunks:,}")
        print(f"   Total tokens: {total_tokens:,}")
        
except Exception as e:
    print(f"❌ Could not query results: {e}")
    print("Make sure to update 'your_warehouse_id' with your actual warehouse ID")

print("\n🎉 Document parsing complete!")
print("\n### Next Steps:")
print("- Run **3_parse_ray.ipynb** for parallel processing")
print("- Run **app.py** to chat with your processed documents")
print("- Check the Delta tables for your processed content")